# StegoPipe — CNN steganalysis (Xu-Net), fully automated

**Two steps:**
1. `Runtime → Change runtime type → GPU (T4)` → Save.
2. `Runtime → Run all`. The E10 result table is printed at the end.

The notebook clones the repository, unzips the 1000-image BOSSBase subset shipped
in it, generates cover/stego pairs with the StegoPipe carriers, trains Xu-Net for
several configurations, and prints and saves the result table. No manual upload or
editing is required.

Reference time on a T4 GPU: about 20–30 minutes per configuration; the default is
three configurations.


## 0. Configuration


In [ ]:
# configurations to run automatically: (carrier, bpp)
CONFIGS = [
    ('lsbm',     0.40),
    ('adaptive', 0.20),
    ('amx',      0.10),
    # ('lsb',    0.40),   # uncomment to add more
    # ('dct',    0.012),
    # ('rdct',   0.010),
]
N_IMAGES = 1000   # number of cover images (max 1000)
EPOCHS   = 30     # increase to 40-50 for a more stable result
BATCH    = 16
SEED     = 0


## 1. Setup: fetch source and data

Set `REPO_URL` to the repository location before running.


In [ ]:
import torch, os, glob, sys
REPO_URL = 'https://github.com/<repository>/stegopipe.git'  # set to the repository URL
print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU (enable a GPU runtime)')
if not os.path.isdir('stegopipe'):
    os.system(f'git clone -q {REPO_URL}')
os.makedirs('data/s1', exist_ok=True); os.makedirs('data/s2', exist_ok=True)
os.system('unzip -o -j -q stegopipe/experiments/bossbase_subset1.zip -d data/s1')
os.system('unzip -o -j -q stegopipe/experiments/bossbase_subset2.zip -d data/s2')
if 'stegopipe' not in sys.path: sys.path.insert(0, 'stegopipe')
PATHS = sorted(glob.glob('data/**/*.png', recursive=True))
print('Cover images found:', len(PATHS))
assert len(PATHS) >= 100, 'No images found - check the unzip step.'


## 2. Definitions (data generation, Xu-Net, train, eval)


In [ ]:
import numpy as np, time
import torch.nn as nn, torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from stegopipe.methods import get_method
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
H = W = 256

def make_pairs(carrier_name, bpp, n_images, seed=0):
    carrier = get_method(carrier_name)
    prng = np.random.default_rng(12345)
    covers, stegos = [], []
    nb_used = None
    for p in PATHS[:n_images]:
        a = np.array(Image.open(p).convert('L'), dtype=np.uint8)
        if a.shape != (H, W):
            y0, x0 = (a.shape[0]-H)//2, (a.shape[1]-W)//2
            a = a[y0:y0+H, x0:x0+W]
        cap = carrier.capacity_bits(a)//8
        nb = int(bpp*H*W/8)
        if nb < 1 or nb > cap:
            continue
        blob = prng.integers(0,256,size=nb,dtype=np.uint8).tobytes()
        try: st = carrier.embed(a, blob)
        except Exception: continue
        covers.append(a); stegos.append(st); nb_used = nb
    return covers, stegos, nb_used

KV = torch.tensor([[-1,2,-2,2,-1],[2,-6,8,-6,2],[-2,8,-12,8,-2],
                   [2,-6,8,-6,2],[-1,2,-2,2,-1]], dtype=torch.float32)/12.0
class XuNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.hpf=nn.Conv2d(1,1,5,padding=2,bias=False)
        self.hpf.weight.data=KV.view(1,1,5,5); self.hpf.weight.requires_grad=False
        self.c1=nn.Conv2d(1,8,5,padding=2);  self.b1=nn.BatchNorm2d(8)
        self.c2=nn.Conv2d(8,16,5,padding=2); self.b2=nn.BatchNorm2d(16)
        self.c3=nn.Conv2d(16,32,1);          self.b3=nn.BatchNorm2d(32)
        self.c4=nn.Conv2d(32,64,1);          self.b4=nn.BatchNorm2d(64)
        self.c5=nn.Conv2d(64,128,1);         self.b5=nn.BatchNorm2d(128)
        self.fc=nn.Linear(128,2)
    def forward(self,x):
        x=self.hpf(x)
        x=F.avg_pool2d(torch.tanh(self.b1(torch.abs(self.c1(x)))),5,2,2)
        x=F.avg_pool2d(torch.tanh(self.b2(self.c2(x))),5,2,2)
        x=F.avg_pool2d(torch.relu(self.b3(self.c3(x))),5,2,2)
        x=F.avg_pool2d(torch.relu(self.b4(self.c4(x))),5,2,2)
        x=torch.relu(self.b5(self.c5(x)))
        x=F.adaptive_avg_pool2d(x,1).flatten(1)
        return self.fc(x)

class Steg(Dataset):
    def __init__(self, cov, ste, ids, aug=False):
        self.cov=cov; self.ste=ste; self.ids=ids; self.aug=aug
    def __len__(self): return len(self.ids)*2
    def __getitem__(self,k):
        i=self.ids[k//2]; lab=k%2
        img=(self.ste[i] if lab else self.cov[i]).astype(np.float32).copy()
        if self.aug:
            if np.random.rand()<.5: img=np.fliplr(img).copy()
            img=np.rot90(img,np.random.randint(4)).copy()
        img=(img-128.0)/64.0
        return torch.from_numpy(img)[None], lab

def auc_score(ys, ps):
    order=np.argsort(ps); ranks=np.empty_like(order,float); ranks[order]=np.arange(1,len(ps)+1)
    npos=int((ys==1).sum()); nneg=int((ys==0).sum())
    if npos==0 or nneg==0: return float('nan')
    return (ranks[ys==1].sum()-npos*(npos+1)/2)/(npos*nneg)

def train_eval(cov, ste, epochs, seed=0):
    rng=np.random.default_rng(seed)
    cov=np.stack(cov); ste=np.stack(ste)
    idx=np.arange(len(cov)); rng.shuffle(idx)
    n=len(idx); ntr,nva=int(.6*n),int(.2*n)
    tr,va,te=idx[:ntr],idx[ntr:ntr+nva],idx[ntr+nva:]
    dl_tr=DataLoader(Steg(cov,ste,tr,aug=True),batch_size=BATCH,shuffle=True,num_workers=2)
    dl_va=DataLoader(Steg(cov,ste,va),batch_size=BATCH)
    dl_te=DataLoader(Steg(cov,ste,te),batch_size=BATCH)
    net=XuNet().to(DEV)
    opt=torch.optim.Adam(net.parameters(),lr=1e-3,weight_decay=1e-4)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,epochs)
    crit=nn.CrossEntropyLoss()
    def run(dl,train):
        net.train() if train else net.eval(); tot=cor=0; Ls=0.0
        with torch.set_grad_enabled(train):
            for x,y in dl:
                x,y=x.to(DEV),y.to(DEV); out=net(x); loss=crit(out,y)
                if train: opt.zero_grad(); loss.backward(); opt.step()
                Ls+=loss.item()*len(y); cor+=(out.argmax(1)==y).sum().item(); tot+=len(y)
        return Ls/tot, cor/tot
    best=0.0
    for ep in range(epochs):
        _,ta=run(dl_tr,True); _,vacc=run(dl_va,False); sch.step()
        if vacc>best: best=vacc; torch.save(net.state_dict(),'best.pt')
        if (ep+1)%10==0 or ep==0: print(f'   ep{ep+1:02d} train={ta:.3f} val={vacc:.3f} (best {best:.3f})')
    net.load_state_dict(torch.load('best.pt')); net.eval()
    ys=[]; ps=[]
    with torch.no_grad():
        for x,y in dl_te:
            p=torch.softmax(net(x.to(DEV)),1)[:,1].cpu().numpy(); ps+=list(p); ys+=list(y.numpy())
    ys=np.array(ys); ps=np.array(ps)
    acc=float(((ps>=.5).astype(int)==ys).mean()); auc=auc_score(ys,ps)
    return acc, auc, len(te)
print('Definitions ready. Device =', DEV)


## 3. Run all configurations


In [ ]:
results=[]
t_all=time.time()
for ci,(carrier,bpp) in enumerate(CONFIGS):
    print(f'\n===== [{ci+1}/{len(CONFIGS)}] {carrier} @ {bpp} bpp =====')
    t0=time.time()
    cov,ste,nb=make_pairs(carrier,bpp,N_IMAGES,SEED)
    if len(cov)<40:
        print(f'   SKIPPED: only {len(cov)} pairs (bpp too high for this carrier capacity).')
        results.append((carrier,bpp,None,None,len(cov))); continue
    print(f'   {len(cov)} cover/stego pairs (payload {nb} B). Training...')
    acc,auc,nte=train_eval(cov,ste,EPOCHS,SEED)
    print(f'   --> acc={acc:.3f}  AUC={auc:.3f}  (test {nte} pairs, {time.time()-t0:.0f}s)')
    results.append((carrier,bpp,acc,auc,nte))
print(f'\nTOTAL TIME: {(time.time()-t_all)/60:.1f} min')


## 4. E10 result table


In [ ]:
E8_AUC={'lsb':'~1.00','lsbm':'~0.998','adaptive':'~0.94','amx':'~0.79','dct':'~0.99','rdct':'~0.97'}
lines=[]
lines.append('| carrier | bpp | Xu-Net acc | Xu-Net AUC | E8 MLP AUC (ref) |')
lines.append('|---------|-----|-----------|-----------|------------------|')
for carrier,bpp,acc,auc,nte in results:
    if acc is None:
        lines.append(f'| {carrier} | {bpp} | (skipped) | (skipped) | {E8_AUC.get(carrier,"-")} |')
    else:
        lines.append(f'| {carrier} | {bpp} | {acc:.3f} | {auc:.3f} | {E8_AUC.get(carrier,"-")} |')
table='\n'.join(lines)
print(table)
with open('E10_xunet_results.md','w') as f: f.write(table+'\n')
print('\nSaved: E10_xunet_results.md')
try:
    from google.colab import files; files.download('E10_xunet_results.md')
except Exception:
    print('(Download E10_xunet_results.md from the Files panel if needed.)')


## 5. Notes

To run other carriers, edit `CONFIGS` in cell 0 and re-run all cells.
To use SR-Net (stronger, slower) instead of Xu-Net, replace the `XuNet` class in cell 2.
